In [ ]:
import sys
print(sys.executable)

In [ ]:
import openslide
print(openslide.__version__)
print(openslide.__library_version__)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

he_path = Path(r"Datos\SB012\SB012\H&E\SB012\SB012.mrxs")

img = Image.open(he_path)

print("Formato:", img.format)
print("Tamaño:", img.size)
print("Modo:", img.mode)

plt.figure(figsize=(8, 12))
plt.imshow(img)
plt.axis("off")
plt.title("H&E preview (2D)")
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image

he_path = Path(r"Datos\SB012\SB012\H&E\SB012\SB012.mrxs")
out_path = he_path.parent / "SB012_HE_preview.png"

img = Image.open(he_path)
img.save(out_path)

print("Guardada en:", out_path)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

png_path = Path(r"Datos\SB012\SB012\H&E\SB012\SB012_HE_preview.png")

img = Image.open(png_path)

plt.figure(figsize=(8, 12))
plt.imshow(img)
plt.axis("off")
plt.title("H&E preview PNG")
plt.show()

In [ ]:
from pathlib import Path

hdr_path = Path(r"Datos\SB012\SB012\SB012 HSI\SB012_M1_002\SB012_M1_nrm.hdr")
print(hdr_path.read_text(encoding="utf-8", errors="ignore"))

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from spectral import envi

base = Path(r"Datos")

hdr_matches = list(base.rglob("SB012_M1_nrm.hdr"))
data_matches = list(base.rglob("SB012_M1_nrm"))

print("HDR encontrados:")
for p in hdr_matches:
    print(" ", p)

print("\nDatos encontrados:")
for p in data_matches:
    print(" ", p)

hdr_path = hdr_matches[0]
data_path = data_matches[0]

img = envi.open(str(hdr_path), str(data_path))
cube = np.array(img.load(), dtype=np.float32)

print("\nShape del cubo:", cube.shape)
print("Metadata keys:", img.metadata.keys())

In [ ]:
wavelengths = np.array([float(w) for w in img.metadata["wavelength"]])

print("Número de bandas:", len(wavelengths))
print("Primera banda:", wavelengths[0])
print("Última banda:", wavelengths[-1])

In [ ]:
def nearest_band(target_nm, wavelengths):
    return int(np.argmin(np.abs(wavelengths - target_nm)))

b_idx = nearest_band(450, wavelengths)
g_idx = nearest_band(550, wavelengths)
r_idx = nearest_band(650, wavelengths)

print("Banda azul :", b_idx, wavelengths[b_idx])
print("Banda verde:", g_idx, wavelengths[g_idx])
print("Banda roja :", r_idx, wavelengths[r_idx])

rgb = cube[:, :, [r_idx, g_idx, b_idx]]

In [ ]:
def percentile_stretch(rgb, p_low=2, p_high=98):
    out = np.zeros_like(rgb, dtype=np.float32)
    for c in range(3):
        band = rgb[:, :, c]
        lo, hi = np.percentile(band, [p_low, p_high])
        out[:, :, c] = np.clip((band - lo) / (hi - lo + 1e-8), 0, 1)
    return out

rgb_show = percentile_stretch(rgb)

plt.figure(figsize=(10, 8))
plt.imshow(rgb_show)
plt.axis("off")
plt.title("SB012 HSI false RGB")
plt.show()

In [ ]:
out_path = hdr_path.parent / "SB012_HSI_falseRGB.png"
plt.imsave(out_path, rgb_show)
print("Guardada en:", out_path)